# 03 · Factor Decay Analysis



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.optimize import curve_fit
%pip install statsmodels scipy

DATA = "/Users/kylechan/Desktop/Microstructure_Factor_Decay Analysis Across_Liquidity_Regimes/data/master_metrics_with_regimes.csv"
df = pd.read_csv(DATA)
df["ts_recv"] = pd.to_datetime(df["ts_recv"])
df = df.sort_values(["symbol", "ts_recv"])

# ── Signals ───────────────────────────────────────────────────────────────────
# Momentum: previous bar's return
df["signal_mom"] = df.groupby("symbol")["return"].shift(1)

# Mean-reversion: negative of 3-bar rolling return (~15 min lookback)
df["signal_rev"] = df.groupby("symbol")["return"].transform(
    lambda x: -x.shift(1).rolling(3).sum()
)

# Forward returns: cumulative sum over next h bars
horizons = [1, 2, 4, 6, 12]  # × 5 min = 5, 10, 20, 30, 60 min

def rolling_fwd_return(series, h):
    """Cumulative return over the next h 5-minute bars."""
    return series.shift(-1).rolling(h).sum().shift(-(h - 1))

for h in horizons:
    df[f"fwd_ret_{h}"] = df.groupby("symbol")["return"].transform(
        lambda s, _h=h: rolling_fwd_return(s, _h)
    )

df = df.dropna()
print(f"Dataset ready: {len(df):,} observations.")


In [ ]:
# Spearman IC at each horizon, split by regime — works for any signal column
def calculate_ic_decay(df, horizons, signal_col="signal_mom"):
    regime_map = {0: "Abundant", 1: "Normal", 2: "Stressed"}
    rows = []
    for regime_id, name in regime_map.items():
        rdf = df[df["regime"] == regime_id]
        ics = [rdf[signal_col].corr(rdf[f"fwd_ret_{h}"], method="spearman")
               for h in horizons]
        rows.append(ics)
    return pd.DataFrame(rows, index=regime_map.values(),
                        columns=[h * 5 for h in horizons])

ic_mom_table = calculate_ic_decay(df, horizons, signal_col="signal_mom")
ic_rev_table = calculate_ic_decay(df, horizons, signal_col="signal_rev")

print("--- Momentum IC ---")
display(ic_mom_table)
print("--- Mean-Reversion IC ---")
display(ic_rev_table)


In [ ]:
# IC decay curves — momentum only (existing chart)
plt.figure(figsize=(12, 6))
colors = {"Abundant": "green", "Normal": "blue", "Stressed": "red"}

for regime in ic_mom_table.index:
    plt.plot(ic_mom_table.columns, ic_mom_table.loc[regime],
             marker="o", linestyle="-", linewidth=2,
             label=regime, color=colors[regime])

plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.title("Momentum Signal Behaviour Across Liquidity Regimes", fontsize=14)
plt.xlabel("Time Horizon (Minutes)", fontsize=12)
plt.ylabel("Information Coefficient (Spearman)", fontsize=12)
plt.legend(title="Liquidity Regime")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Newey-West HAC significance + exponential decay fit for momentum factor
def exp_decay(h, alpha, beta):
    return alpha * np.exp(-beta * h)

horizons_min  = np.array([5, 10, 20, 30, 60])
horizons_cols = [f"fwd_ret_{h}" for h in horizons]
decay_results = []

for regime_id in [0, 1, 2]:
    regime_name = {0: "Abundant", 1: "Normal", 2: "Stressed"}[regime_id]
    rdf = df[df["regime"] == regime_id].dropna(subset=["signal_mom"] + horizons_cols)

    if len(rdf) < 30:
        print(f"⚠️  Skipping {regime_name}: only {len(rdf)} rows.")
        continue

    ics = [rdf["signal_mom"].corr(rdf[col], method="spearman") for col in horizons_cols]

    nw_stats = {}
    for h, col in zip(horizons_min, horizons_cols):
        X = sm.add_constant(rdf["signal_mom"])
        y = rdf[col]
        try:
            res = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": max(1, h // 5)})
            nw_stats[h] = (res.tvalues.iloc[1], res.pvalues.iloc[1])
        except Exception as e:
            print(f"  NW failed {regime_name} h={h}: {e}")
            nw_stats[h] = (np.nan, np.nan)

    try:
        popt, _ = curve_fit(exp_decay, horizons_min, ics,
                            p0=[0.05, 0.1], bounds=([0, 0], [1, 2]), maxfev=10000)
        alpha_fit, beta = popt
        half_life = np.log(2) / beta
    except Exception as e:
        print(f"  curve_fit failed {regime_name}: {e}")
        alpha_fit, beta, half_life = np.nan, np.nan, np.nan

    row = {"Regime": regime_name,
           "Initial IC (alpha)": round(alpha_fit, 4),
           "Decay Rate (beta)":  round(beta, 4),
           "Half-Life (mins)":   round(half_life, 1)}
    for h in horizons_min:
        t, p = nw_stats[h]
        row[f"t-stat {h}m"] = round(t, 2)
        row[f"p-val {h}m"]  = round(p, 4)
    decay_results.append(row)

decay_summary = pd.DataFrame(decay_results)
print("\n--- Exponential Decay Results (HAC standard errors) ---")
display(decay_summary)


In [ ]:
# PRIMARY VISUALISATION: side-by-side IC heatmaps — momentum vs mean-reversion
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

sns.heatmap(ic_mom_table.round(4), annot=True, fmt=".4f",
            cmap="RdYlGn", center=0, linewidths=0.5, ax=axes[0])
axes[0].set_title("Momentum IC by Regime × Horizon")
axes[0].set_xlabel("Horizon (Minutes)")

sns.heatmap(ic_rev_table.round(4), annot=True, fmt=".4f",
            cmap="RdYlGn", center=0, linewidths=0.5, ax=axes[1])
axes[1].set_title("Mean-Reversion IC by Regime × Horizon")
axes[1].set_xlabel("Horizon (Minutes)")

plt.suptitle("Factor Behaviour Across Liquidity Regimes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# CROSSOVER ANALYSIS — at which horizon does each regime switch dominant signal?
print("Signal crossover horizons by regime:\n")
for regime in ["Stressed", "Normal", "Abundant"]:
    print(f"--- {regime} ---")
    for h in [h * 5 for h in horizons]:
        mom_ic = ic_mom_table.loc[regime, h] if regime in ic_mom_table.index else 0
        rev_ic = ic_rev_table.loc[regime, h] if regime in ic_rev_table.index else 0
        dominant = "MOM" if mom_ic > rev_ic else "REV"
        print(f"  {h:>3} min: momentum={mom_ic:+.4f}  mean-rev={rev_ic:+.4f}  → {dominant} dominates")
    print()


In [ ]:
print("""
Key findings:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Abundant  Both signals ≈ 0 at all horizons.
          Consistent with high market efficiency — alpha
          is immediately arbitraged away in liquid markets.

Normal    Momentum positive at 5min, reverses to negative
          by 10-30min. Mean-reversion correspondingly
          negative early, positive later. Classic
          short-term momentum → medium-term reversal.

Stressed  Momentum builds from 5min → 20min (delayed
          price discovery). Mean-reversion signal shows
          the complementary pattern. Illiquid markets
          incorporate information slowly, creating a
          persistent but lagged momentum effect.

Crossover The horizon at which momentum stops dominating
          and mean-reversion takes over shifts by regime —
          earlier in liquid markets, later (or absent) in
          stressed markets. This crossover is directly
          actionable for regime-aware execution algorithms.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
